# GuitarTranscriber - training on GOAT + GuitarSet + Guitar-TECHS

DI audio -> note events (pitch, onset, offset), causal, streaming, **pitch-major**:
one onset/sustain row per pitch, no string identity (per-string heads retired).

**Read before running:**

1. **Three corpora, one stream**. GOAT is the reference domain.
   GuitarSet is acoustic and **train-only** (`build_splits` enforces it). Guitar-TECHS'
   continuous takes join the pool, its chords/notes become a sample bank pasted into
   windows, and `P3_music` is a second validation axis.
2. **Three validation axes, never merged.** `val/*` (GOAT holdout), `valgt/*`
   (Guitar-TECHS P3_music - unseen player+guitar+chain), `trainset/*` (seen GOAT,
   catches memorization). Compare each axis against its own history, never across
   axes or across runs whose reference set changed.
3. **No velocity anywhere** - MIDI writers emit a constant.
4. Validation is split **by recording**, never by random window - the GRU can
   memorize note sequences, and a random split would hide exactly that.

In [ ]:
import json, math, os, sys, time, warnings
warnings.filterwarnings("ignore")

import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

# this repo, found from the kernel's working directory (train/ or the root)
PROJECT = os.path.abspath(os.getcwd())
if os.path.basename(PROJECT) == "train":
    PROJECT = os.path.dirname(PROJECT)
assert os.path.isfile(os.path.join(PROJECT, "train", "config.py")), \
    "start the kernel inside the repo: " + PROJECT
TRAIN_DIR = os.path.join(PROJECT, "train")
sys.path.insert(0, TRAIN_DIR)

import augment, corpus, goat, guitarset, guitartechs, metrics
from config import CONFIG, Config
from data import WindowComposer, build_splits, silent_regions, compute_norm_stats
from model import GuitarTranscriber, transcription_loss

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

cfg = CONFIG  # conv2d trunk; conv1d was A/B'd and retired (see config.py)
# cfg = Config(causal=False, time_kernel=5)   # the offline twin (model.py)
# WARM_START fine-tunes the twin from the converted causal checkpoint
# (train/offline_init.py); False cold-starts it on the from-scratch recipe
WARM_START = not cfg.causal

print(f"device        {DEVICE}")
print(f"trunk         {cfg.trunk}")
print(f"sample rate   {cfg.sample_rate} Hz, hop {cfg.hop_length} = {cfg.frame_ms:.2f} ms "
      f"({cfg.frame_rate:.1f} fps)")
print(f"window        {cfg.window_seconds}s = {cfg.window_frames} frames "
      f"({cfg.window_samples} samples), loss from frame {cfg.loss_cutoff}")
print(f"labels        {cfg.n_pitches} pitch rows (MIDI {cfg.min_pitch}..{cfg.max_pitch}), "
      f"pitch-major - no string identity")
print(f"dataset draw  {corpus.DATASET_WEIGHT}")
print(f"causal        {cfg.causal}  time kernel {cfg.time_kernel}  warm start {WARM_START}")

## 1. Cache

One cache root per dataset (`F:\datasets\<name>\cache_24k`), one merged loader. Each
ingest is idempotent - re-running skips what's already on disk, and audio decode
(the expensive half) is version-independent, so an event-format change never
re-decodes audio. First build is ~5 minutes total; after that, seconds.

Three things the ingests measure rather than assume, and print either way:

- **GOAT** drops recordings whose `alignment_f_measure_fine` is below `MIN_ALIGN_F`,
  and rejects amp renders whose lag is unstable across segments.
- **GuitarSet** measures inter-string bleed per channel and prints the verdict on
  the label-dropping submix - the channels are string-*dominant*, not isolated.
- **Guitar-TECHS** prints what its dirt filter removed: hanging note-ons, sub-50 ms
  notes (floored, not dropped), and notes outside fret 0-24 on their labeled string.

In [ ]:
MIN_ALIGN_F = 0.5
DATASETS = ("goat", "guitarset", "guitartechs")

# idempotent: a stale cache RAISES rather than serving old labels (versioned event
# tables, a manifest fingerprinted on sample rate), so recovery is "rebuild whichever complained"
for name, builder in (("goat", lambda: goat.build_cache(min_align_f=MIN_ALIGN_F,
                                                        verbose=False)),
                      ("guitarset", lambda: guitarset.build_cache(verbose=False)),
                      ("guitartechs", lambda: guitartechs.build_cache(verbose=False))):
    out = corpus.cache_dir(corpus.DATASET_ROOTS[name], cfg)
    try:
        corpus.read_manifest(out, cfg, dataset=name)
    except (FileNotFoundError, RuntimeError) as e:
        print(f"{name}: rebuilding ({str(e).splitlines()[0][:110]})")
        t0 = time.perf_counter()
        builder()
        print(f"  built in {time.perf_counter() - t0:.0f}s")

manifest, bank = corpus.load_corpus(DATASETS, cfg)

import collections
print()
for key in ("dataset", "player", "guitar"):
    c = collections.Counter(e.get(key, "?") for e in manifest)
    print(f"{key:8s} {dict(c)}")
print(f"bank     {len(bank)} stems, {sum(b['n_units'] for b in bank)} units, "
      f"{sum(b['seconds'] for b in bank) / 3600:.2f} h, "
      f"{sum(b['n_events'] for b in bank)} events "
      f"(mean {100 * np.mean([b['clean_head_frac'] for b in bank]):.0f}% clean heads - "
      f"the rest get their lead weight-zeroed at paste time)")
print(f"renders  {sum(len(e.get('renders', [])) for e in manifest)} available, "
      f"{sum(e.get('renders_rejected', 0) for e in manifest)} rejected for unstable lag")

## 2. Split

`build_splits` assigns every recording to exactly one axis, reading the rules off the
manifest rather than off a list somebody maintains by hand:

| axis | material | what it isolates |
|---|---|---|
| `val/*` | GOAT holdout, 3% by duration, whole (player, guitar) groups | held-out guitars, same corpus style |
| `valgt/*` | Guitar-TECHS `P3_music` | unseen player + guitar + recording chain, electric DI |
| `trainset/*` | a fixed slice of seen GOAT recordings | memorization (the anticipation watchpoint) |

The primary split is drawn from **GOAT alone**, deliberately: that is the reference set
every earlier run was measured against, and mixing other corpora into the draw would move
it. Comparing F1s across runs whose reference set changed is the mistake
`eval_checkpoints.py` exists to prevent.

`train_only` recordings (all of GuitarSet - acoustic guitars, a different instrument
family from the product's input) can never reach a validation axis; the assert in
`build_splits` says so out loud.

Watch the composition it prints. GOAT validation lands as one or two guitars, so that
number is a *harder* test than average - read it as a lower bound rather than an estimate.

In [ ]:
VAL_FRAC = 0.10
SPLIT_SEED = 0

splits = build_splits(manifest, val_frac=VAL_FRAC, seed=SPLIT_SEED, primary="goat")
train_entries = splits["train"]
val_entries   = splits["val"]
valgt_entries = splits["valgt"]

## 3. Datasets

Every training window is composed in five steps. 1-3 multiply content, 4-5 multiply
tone and level. **The rule every step obeys: the result must stay a plausible DIRECT
INPUT** - the product transcribes the pickup output, so anything else spends capacity
on a signal the model will never be handed.

| step | what | label transform |
|---|---|---|
| **1 BASE** | continuous window, dataset-weighted; pitch shift within the recording's clamp; channel-gain remix on multi-channel sources | `pitch += n, times x 2^(-n/12)` |
| **2 OVERLAY (p=0.3)** | second window at -9..0 dB | UNION |
| **3 PASTE (p=0.3)** | 1-4 bank units at random offsets, -9..+3 dB; 10% of windows are bank-only | UNION + seam rules below |
| 4 TIMBRE | pickup comb/resonance, cable EQ, gentle saturation | none |
| 5 LEVEL/NOISE | gain +-6 dB, hum/hiss/bleed | none |

Every source is normalized to `cfg.norm_level_dbfs` before step 1, in validation too -
it's the input convention the host honours (`gdi2m --gain`), not an augmentation.

Pitch shift is what delivers tempo variation (no phase vocoder, no transient smear);
`p_stretch` is off by default for that reason.

Paste seam rules, each preventing one specific label lie: fade **in** before the
unit's first onset (a raw cut mid-ring is an unlabeled transient); fade **out**
wherever it's cut, with offsets truncated to match; **zero the label weight** over a
pasted head unless the lead is provably silent; density is a knob and **0 is a value**
(a bank-only window may draw nothing).

Overlay and paste are legal because labels are pitch-major - every note arrives with
its own label, so the union is truthful. GuitarSet's label-*dropping* submix stays
refused (the ingest's bleed measurement failed its own gate); the gain remix is safe
and ships.

**Listen to what this produces**: `python train/preview_batch.py` plays the composed
audio in one ear and the label tensor in the other, and prints the recipe.

In [ ]:
BATCH_SIZE      = 8      # 4s windows at 256x750 are activation-heavy; drop to 4 on OOM
STEPS_PER_EPOCH = 1500
NUM_WORKERS     = 4

# GOAT DI only, only from silent stretches - hum/room/handling noise without adding
# note content the label tensor doesn't know about
print("building bleed pool from silent regions...")
bleed_pool = []
for e in [x for x in train_entries if x["dataset"] == "goat"][:40]:
    bleed_pool += silent_regions(e, cfg, min_sec=0.5, max_regions=6)
print(f"  {len(bleed_pool)} segments, "
      f"{sum(len(s) for s in bleed_pool) / cfg.sample_rate:.0f}s total")

# no augmentation kwargs here, deliberately: the recipe is WindowComposer's own
# defaults, so this cell and preview_batch.py build the composer bare and can never
# drift apart - changing data.py changes both at once
train_ds = WindowComposer(train_entries, cfg, train=True, bank=bank,
                          steps_per_epoch=STEPS_PER_EPOCH * BATCH_SIZE,
                          bleed_pool=bleed_pool, seed=1)

# validation composes nothing - raw signal, deterministic windows, no augmentation,
# the only way two checkpoints' numbers describe the same question. Level
# normalization still applies both sides: it's the input convention, not an augmentation
val_ds   = WindowComposer(val_entries, cfg, train=False)
valgt_ds = WindowComposer(valgt_entries, cfg, train=False)

# a fixed slice of TRAIN recordings, walked like validation - the memorization
# instrument. Read its onset delay against val's, never on its own.
_probe_all = WindowComposer([e for e in train_entries if e["dataset"] == "goat"],
                            cfg, train=False)
_stride = max(1, len(_probe_all) // len(val_ds))
train_probe_ds = torch.utils.data.Subset(
    _probe_all, list(range(0, len(_probe_all), _stride))[:len(val_ds)])

frac = {}
for e, w in zip(train_ds.entries, train_ds.weights):
    frac[e["dataset"]] = frac.get(e["dataset"], 0.0) + float(w)
print("window draw probability: "
      + "  ".join(f"{k} {100 * v:.1f}%" for k, v in sorted(frac.items())))
print(f"bank units {len(train_ds.bank_index)}  |  p_render {train_ds.p_render} "
      f"(renders listed in the manifest: "
      f"{sum(len(e.get('renders', [])) for e in train_entries)}, unread)")

print(f"\nsources normalized to {cfg.norm_level_dbfs:+.1f} dBFS active level; "
      f"composition (WindowComposer defaults):")
_s = train_ds.settings
_keys = sorted(_s)
for _a in range(0, len(_keys), 3):
    print("  " + "   ".join(f"{k}={_s[k]}" for k in _keys[_a:_a + 3]))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
                          pin_memory=True, drop_last=True, persistent_workers=NUM_WORKERS > 0)
# validation loaders run in-process (num_workers=0): with workers>0, every validation
# spawned six fresh python processes re-importing torch - tens of seconds of pure
# startup every 1000 steps, plus disk contention with the train workers
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, num_workers=0, pin_memory=True)
valgt_loader = DataLoader(valgt_ds, batch_size=BATCH_SIZE, num_workers=0, pin_memory=True)
train_probe_loader = DataLoader(train_probe_ds, batch_size=BATCH_SIZE, num_workers=0,
                                pin_memory=True)

print(f"\ntrain {len(train_ds)} windows/epoch ({len(train_loader)} steps) | "
      f"val {len(val_ds)} | valgt {len(valgt_ds)} | "
      f"train probe {len(train_probe_ds)} from {len(_probe_all.entries)} seen recordings")

## 4. Model and input normalization

`set_norm_stats` is not optional. The front-end emits log-power in the tens; feeding that
raw spends the first few thousand steps learning a rescale the optimizer should never have
been asked to learn.

In [ ]:
model = GuitarTranscriber(cfg).to(DEVICE)
print(f"parameters {sum(p.numel() for p in model.parameters()) / 1e6:.2f} M ({cfg.trunk})")

# fingerprint covers only the fields that shape the front end's output scale (not
# the trunk variant), measured on GOAT raw DI with every composition step OFF - the
# stats must describe what the product sees, not what augmentation manufactures
FRONTEND_KEYS = ("sample_rate", "hop_length", "n_ffts", "window_peaks",
                 "f_min", "bins_per_octave", "n_bins", "norm_level_dbfs",
                  "n_harmonics")
STATS_PATH = os.path.join(goat.CACHE_DIR, "norm_stats.json")
cfg_fp = json.loads(json.dumps({k: getattr(cfg, k) for k in FRONTEND_KEYS}))
norm = None
if os.path.exists(STATS_PATH):
    saved = json.load(open(STATS_PATH))
    if saved.get("cfg") == cfg_fp:
        norm = saved
    else:
        print("norm stats were computed under a different front-end - recomputing")
if norm is None:
    stats_ds = WindowComposer(
        [e for e in train_entries if e["dataset"] == "goat"], cfg, train=True,
        steps_per_epoch=192, p_pitch=0.0, p_noise=0.0, p_render=0.0,
        p_overlay=0.0, p_paste=0.0, p_bank_only=0.0, p_chan_gain=0.0,
        p_comb=0.0, p_resonance=0.0, p_drive=0.0, p_eq=0.0,
        gain_db=(0.0, 0.0), seed=7)
    mean, std, e_std = compute_norm_stats(model, stats_ds, batch_size=8, device=DEVICE)
    norm = dict(shape_mean=mean, shape_std=std, e_rel_std=e_std, cfg=cfg_fp)
    json.dump(norm, open(STATS_PATH, "w"), indent=1)

model.set_norm_stats(norm["shape_mean"], norm["shape_std"], norm["e_rel_std"])
print(f"shape mean {norm['shape_mean']:.3f} std {norm['shape_std']:.3f} | "
      f"e_rel std {norm['e_rel_std']:.3f}")

In [ ]:
# a warm-started twin is fine-tuned: the from-scratch recipe would walk it off
# the causal point and re-traverse training, re-adding the optimizer-luck
# term the exact warm start removes
if WARM_START:
    LR, WARMUP_STEPS, TOTAL_STEPS = 1e-4, 500, 30_000
else:
    LR, WARMUP_STEPS, TOTAL_STEPS = 3e-4, 2000, 100_000    # cosine shaped to when val plateaus
WEIGHT_DECAY  = 1e-2
# L1 on the GRU gate: prices how much unbounded memory the model gets, since that's
# the one thing here that can memorize note SEQUENCES. L1 not weight decay - the
# wanted answer is sparse (some channels need memory for a decayed note still
# ringing, most don't), a quadratic pull would shrink all of them a little instead.
# Watch train/gate_mean: climbing back toward ~0.65 with most channels open means
# this is too weak; collapsing toward 0 means too strong.
GATE_L1       = 0.005

# decay on weight matrices only - biases, norm scales, the freq position embedding,
# the GRU gate and the attention's relative bias all parameterize offsets/scales the
# model should be free to hold
NO_DECAY = ("bias", "norm", "freq_embed", "rnn_gate", "rel_bias")
decay, no_decay = [], []
for name_, p_ in model.named_parameters():
    if any(k in name_ for k in NO_DECAY):
        no_decay.append(p_)
    else:
        decay.append(p_)
opt = torch.optim.AdamW(
    [{"params": decay, "weight_decay": WEIGHT_DECAY},
     {"params": no_decay, "weight_decay": 0.0}],
    lr=LR, betas=(0.9, 0.99))
print(f"decayed params {sum(p.numel() for p in decay)/1e6:.2f} M | "
      f"undecayed {sum(p.numel() for p in no_decay)/1e6:.3f} M")

def lr_at(step):
    if step < WARMUP_STEPS:
        return step / max(1, WARMUP_STEPS)
    p = (step - WARMUP_STEPS) / max(1, TOTAL_STEPS - WARMUP_STEPS)
    return 0.02 + 0.98 * 0.5 * (1 + math.cos(math.pi * min(p, 1.0)))

sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_at)
print(f"AdamW lr {LR}, warmup {WARMUP_STEPS} steps, cosine to {TOTAL_STEPS}"
      f"{'  (fine-tune)' if WARM_START else ''}")
print(f"rnn dropout {cfg.rnn_dropout} | gate L1 {GATE_L1} "
      f"(gate starts at {cfg.rnn_gate_init}; watch train/gate_mean)")

## 5. Forward

`forward_batch` is `model(wav)` spelled out - front end, level split, trunk. No
augmentation lives here: every perturbation is on the waveform in `WindowComposer`,
so `preview_batch` can play exactly what trains.

In [ ]:
USE_AMP = True   # bf16 autocast; TF32 already covers conv/matmul, this buys activation memory


def amp():
    return torch.autocast(DEVICE, dtype=torch.bfloat16,
                          enabled=USE_AMP and DEVICE == "cuda")


def forward_batch(model, wav):
    """model(wav) spelled out. No augmentation here - it's all on the waveform in
    WindowComposer, so preview_batch can play exactly what trains."""
    shape, e_rel = model.level(model.spec(wav))
    return model.forward_features(shape, e_rel)


@torch.no_grad()
def run_validation(model, loader, max_batches=None):
    model.eval()
    sums, counts, losses = {}, {}, []
    for i, (wav, tgt) in enumerate(loader):
        if max_batches and i >= max_batches:
            break
        wav = wav.to(DEVICE, non_blocking=True)
        tgt = {k: v.to(DEVICE, non_blocking=True) for k, v in tgt.items()}
        with amp():
            out = forward_batch(model, wav)
        out = {k: v.float() for k, v in out.items()}
        losses.append(float(transcription_loss(out, tgt, cfg)["total"]))
        for k, v in metrics.evaluate(out, tgt, cfg, tol_ms=50.0).items():
            # seed the key even when NaN, or the log line KeyErrors as long as the
            # model predicts nothing
            sums.setdefault(k, 0.0)
            counts.setdefault(k, 0)
            if v == v:
                sums[k] += v
                counts[k] += 1
    model.train()
    return {"val_loss": float(np.mean(losses)),
            **{k: (sums[k] / counts[k] if counts[k] else float("nan")) for k in sums}}


def free_interrupted_graph():
    '''Drop every reference an interrupted step leaves behind, so
    torch.cuda.empty_cache() can actually free the activation graph (an interrupt
    between forward/backward leaves it referenced by both surviving globals and
    IPython's stored traceback).'''
    import gc
    for name in ("out", "losses", "total", "gate_l1", "wav", "tgt", "gnorm",
                 "spec", "shape", "e_rel", "batched", "streamed", "chunks",
                 "o", "x"):
        globals().pop(name, None)

    sys.last_traceback = sys.last_value = sys.last_type = None
    if hasattr(sys, "last_exc"):
        sys.last_exc = None
    try:
        ip = get_ipython()
        for name in ("_", "__", "___", "_i", "_ii", "_iii"):
            ip.user_ns.pop(name, None)
        ip.user_ns.get("Out", {}).clear()
    except NameError:
        pass

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        print(f"cuda allocated {torch.cuda.memory_allocated() / 2**30:5.2f} GiB | "
              f"reserved {torch.cuda.memory_reserved() / 2**30:5.2f} GiB")

## 5b. Compile the trunk

`torch.compile` on the conv stack only - a homogeneous pile of small convs where
per-kernel launch overhead dominates. The GRU stays eager (cuDNN RNNs graph-break
under dynamo) and the compiled path is used only while training; eval, streaming
parity and the drift check take the eager path since a live session feeds one frame
at a time.

Shapes must stay static for this to pay off (`drop_last=True` on the train loader).
Warmup runs here on a synthetic batch so a backend failure shows up now, not at step
1. The CPU backend additionally needs `cl` on PATH.

In [ ]:
USE_COMPILE = True

if USE_COMPILE:
    model.compile_trunk()
    model.train()
    try:
        _t0 = time.perf_counter()
        # pitch-major target: (N, n_pitches, T) - a compile check that fails at
        # step 0 is only worth having if it fails for a real reason
        _w = torch.randn(BATCH_SIZE, cfg.window_samples, device=DEVICE) * 0.05
        _z = torch.zeros(BATCH_SIZE, cfg.n_pitches, cfg.window_frames, device=DEVICE)
        _tg = {"onset": _z, "sustain": _z,
               "weight": torch.ones_like(_z)}
        with amp():
            _o = forward_batch(model, _w)
        _o = {k: v.float() for k, v in _o.items()}
        transcription_loss(_o, _tg, cfg)["total"].backward()
        model.zero_grad(set_to_none=True)
        print(f"trunk compiled and warmed in {time.perf_counter() - _t0:.0f}s")
    except Exception as exc:
        model._trunk_c = None
        print(f"torch.compile failed, continuing eager: "
              f"{type(exc).__name__}: {str(exc)[:300]}")
    finally:
        for _n in ("_w", "_z", "_tg", "_o"):
            globals().pop(_n, None)
        free_interrupted_graph()
else:
    print("compile disabled")

In [ ]:
# load a checkpoint. "resume" continues that exact run (needs *_last.pt for optimizer
# state, and an IDENTICAL config - mixing models is refused). "restart" warm-starts a
# NEW run from the weights alone: fresh optimizer/schedule/run name, config allowed to
# differ (and printed). If the kernel is still alive from an interrupted run, skip this
# cell - re-running the training cell below already resumes in place.
#
# Run names encode the data regime (`goat_*` = GOAT alone, `gdi2m_*` = three-corpus) -
# val/* is comparable across both since it's the same GOAT holdout, nothing else is.
# the converted causal checkpoint (carries the twin's cfg and its provenance):
#   python train/offline_init.py --verify --ckpt checkpoints/<causal>_best.pt \
#     --out checkpoints/offline_twin_init.pt
TWIN_INIT = os.path.join(PROJECT, "checkpoints", "offline_twin_init.pt")
RESUME_FROM = TWIN_INIT if WARM_START else ""
RESUME_MODE = "restart"          # "resume" | "restart"

if RESUME_FROM:
    assert os.path.exists(RESUME_FROM), \
        "converted twin not found at %s - run train/offline_init.py first" % RESUME_FROM

if not RESUME_FROM:
    print("RESUME_FROM not set - the training cell will start a fresh run")
else:
    assert RESUME_MODE in ("resume", "restart"), RESUME_MODE
    from config import HOST_FIELDS, model_fields
    from torch.utils.tensorboard import SummaryWriter

    _ck = torch.load(RESUME_FROM, map_location=DEVICE, weights_only=False)
    _ck_cfg = json.loads(json.dumps(_ck["cfg"]))
    _cur_cfg = json.loads(json.dumps(cfg.to_dict()))
    model.load_state_dict(_ck["model"])

    if RESUME_MODE == "resume":
        assert "opt" in _ck, "need a *_last.pt - *_best.pt carries no optimizer state"
        # model_fields, not the whole dict (same filter export_onnx.py applies) -
        # HOST_FIELDS change no weight, shape or label, only where the decoder cuts.
        # Host drift is printed since everything logged from here decodes with the
        # current constants.
        assert model_fields(_ck_cfg) == model_fields(_cur_cfg), \
            "checkpoint was trained under a DIFFERENT model config - " \
            "resuming would mix models. Use RESUME_MODE='restart' " \
            "to warm start from these weights."
        for _k in sorted(HOST_FIELDS):
            if _ck_cfg.get(_k) != _cur_cfg.get(_k):
                print(f"  host {_k}: checkpoint {_ck_cfg.get(_k)!r} "
                      f"-> now {_cur_cfg.get(_k)!r}")
        opt.load_state_dict(_ck["opt"])
        sched.load_state_dict(_ck["sched"])
        norm = _ck["norm"]
        model.set_norm_stats(norm["shape_mean"], norm["shape_std"], norm["e_rel_std"])
        step = int(_ck["step"])

        # recover best_f1 from _best.pt so a resume can't clobber a better checkpoint
        best_f1 = -1.0
        _bp = RESUME_FROM.replace("_last.pt", "_best.pt")
        if os.path.exists(_bp):
            best_f1 = float(torch.load(_bp, map_location="cpu", weights_only=False)
                            .get("val", {}).get("onset_f1", -1.0))

        RUN_NAME = os.path.basename(RESUME_FROM)[:-len("_last.pt")]
        writer = SummaryWriter(os.path.join(PROJECT, "runs", RUN_NAME))  # appends
        print(f"resumed {RUN_NAME} at step {step} (best onset F1 {best_f1:.3f}) - "
              f"now run the training cell")

    else:
        _parent = os.path.basename(RESUME_FROM)
        for _suf in ("_last.pt", "_best.pt"):
            if _parent.endswith(_suf):
                _parent = _parent[:-len(_suf)]

        # config may differ here - host fields change no weight/shape/label, MODEL
        # fields do, and a warm start under a changed one is deliberate, so print both
        for _k in sorted(set(_ck_cfg) | set(_cur_cfg)):
            if _ck_cfg.get(_k) != _cur_cfg.get(_k):
                _tag = "host " if _k in HOST_FIELDS else "MODEL"
                print(f"  cfg {_tag} {_k}: checkpoint {_ck_cfg.get(_k)!r} "
                      f"-> now {_cur_cfg.get(_k)!r}")

        # norm stats belong to the front end, not the run - reuse the checkpoint's
        # only if the front end is unchanged, or they'd rescale the input wrongly
        if _ck["norm"].get("cfg") == cfg_fp:
            norm = _ck["norm"]
        else:
            print("  front end differs - keeping the norm stats computed above, "
                  "not the checkpoint's")
        model.set_norm_stats(norm["shape_mean"], norm["shape_std"], norm["e_rel_std"])

        # fresh optimizer and schedule: clearing opt.state drops Adam moments that
        # describe a loss surface these weights are no longer being walked on
        opt.state.clear()
        for _g in opt.param_groups:
            _g["initial_lr"] = LR
        sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_at)

        step, best_f1 = 0, -1.0
        _kind = cfg.trunk if cfg.causal else f"{cfg.trunk}_offline"
        RUN_NAME = time.strftime(f"gdi2m_{_kind}_%m%d_%H%M")
        writer = SummaryWriter(os.path.join(PROJECT, "runs", RUN_NAME))
        # provenance in the run itself, so a warm-started curve isn't mistaken later
        # for one trained from scratch
        writer.add_text("warm_start",
                        f"{RUN_NAME} warm started from {_parent} "
                        f"(step {int(_ck['step'])}), weights only", 0)
        print(f"warm start: {_parent} step {int(_ck['step'])} -> new run "
              f"{RUN_NAME} at step 0, lr schedule restarted over {TOTAL_STEPS} "
              f"steps - now run the training cell")

    del _ck

## 6. Train

- `onset_f1` - pitch + timing, the headline.
- `onset_delay_ms` - **signed detection lag over matched notes.** The 50 ms match
  tolerance makes F1 blind to lateness, so this is co-headline with F1, not a detail.
- **`VALGT` / `valgt/*`** - Guitar-TECHS P3_music: unseen player+guitar+chain,
  electric DI. Closest axis to the product's real situation. Read it against its own
  history over time, never against `val` (different reference set).
- **`SEEN` / `trainset/*`** - held-in GOAT. Read its `onset_delay_ms` against val's:
  a NEGATIVE seen-audio delay is the cheapest memorization indicator this setup has,
  and the print flags it.
- `note_f1_offset` / `offset_cost` - matching that also requires the release to land.
  Note-off is the SUSTAIN head at decode time, debounced by `off_hold_ms`.
- `frame_activity_f1` - sustain roll vs target, cellwise.
- `duration_ratio` - predicted note length over reference; ~1.0 is right, well below
  reads torn, well above reads droning.
- `train/gate_mean`, `train/gate_open_frac` - how much unbounded GRU memory the heads
  are actually using; `GATE_L1` and `rnn_dropout` are the levers.

Labels are weighted by confidence in the loss (`target['weight']`, from `align_f` and
the composer's per-source merge); zero-weight cells are provably invisible to the loss.

**Re-running this cell resumes, it does not restart** - `step`/`best_f1`/`writer` are
created only on the first run.

**On VRAM after an interrupt:** `torch.cuda.empty_cache()` only frees what nothing
references, and an interrupt mid-step leaves the activation graph referenced by both
surviving globals and IPython's stored traceback. The loop catches
`KeyboardInterrupt` and calls `free_interrupted_graph()` in a `finally` for this
reason; call it by hand any time VRAM looks wrong.

In [ ]:
from torch.utils.tensorboard import SummaryWriter

CKPT_DIR   = os.path.join(PROJECT, "checkpoints")
LOG_EVERY  = 50
VAL_EVERY  = 1000
CKPT_EVERY = 2000
os.makedirs(CKPT_DIR, exist_ok=True)

free_interrupted_graph()

# re-running this cell RESUMES: step/best_f1/writer are only created the first time,
# so resetting step here while leaving opt/sched untouched would desync the schedule
# from the logging and checkpoint cadence
if "step" not in globals():
    step, best_f1 = 0, -1.0
    _kind = cfg.trunk if cfg.causal else f"{cfg.trunk}_offline"
    RUN_NAME = time.strftime(f"gdi2m_{_kind}_%m%d_%H%M")
    writer = SummaryWriter(os.path.join(PROJECT, "runs", RUN_NAME))
    print(f"new run {RUN_NAME}")
elif step == 0:
    # the cell above set these up: a warm start counted as its own run from step 0
    print(f"starting {RUN_NAME} at step 0")
else:
    print(f"resuming {RUN_NAME} at step {step} (best onset F1 {best_f1:.3f})")

model.train()
t_start, step_start, running = time.perf_counter(), step, {}

try:
    while step < TOTAL_STEPS:
        for wav, tgt in train_loader:
            wav = wav.to(DEVICE, non_blocking=True)
            tgt = {k: v.to(DEVICE, non_blocking=True) for k, v in tgt.items()}

            with amp():
                out = forward_batch(model, wav)
            out = {k: v.float() for k, v in out.items()}
            losses = transcription_loss(out, tgt, cfg)
            # a MODEL penalty, not a data term, so it lives here rather than inside
            # transcription_loss, which stays a pure function of tensors
            gate_l1 = model.gate_penalty()
            total = losses["total"] + GATE_L1 * gate_l1

            opt.zero_grad(set_to_none=True)
            total.backward()
            gnorm = float(torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0))
            opt.step()
            sched.step()
            step += 1

            for k, v in losses.items():
                running[k] = running.get(k, 0.0) + float(v)
            running["gate"] = running.get("gate", 0.0) + float(gate_l1)
            # drop the graph before the next iteration allocates another one
            del out, losses, total, gate_l1, wav, tgt

            if step % LOG_EVERY == 0:
                for k, v in running.items():
                    writer.add_scalar(f"train/{k}", v / LOG_EVERY, step)
                writer.add_scalar("train/lr", sched.get_last_lr()[0], step)
                writer.add_scalar("train/grad_norm", gnorm, step)
                # a mean hides the shape: 40 wide-open channels is a different model
                # from 384 half-open ones
                with torch.no_grad():
                    writer.add_scalar("train/gate_mean",
                                      float(model.rnn_gate.abs().mean()), step)
                    writer.add_scalar("train/gate_open_frac",
                                      float((model.rnn_gate.abs() > 0.5).float().mean()),
                                      step)
                    # the backward gate starts at 0; if it stays there the
                    # branch is unused and "causality is free" would be a
                    # training failure, not a finding
                    if model.rnn_bwd is not None:
                        writer.add_scalar("train/gate_bwd_mean",
                                          float(model.rnn_gate_bwd.abs().mean()), step)
                        writer.add_scalar("train/gate_bwd_open_frac",
                                          float((model.rnn_gate_bwd.abs() > 0.5)
                                                .float().mean()), step)
                        writer.add_scalar(
                            "train/gate_bwd_over_fwd",
                            float(model.rnn_gate_bwd.abs().mean()
                                  / model.rnn_gate.abs().mean().clamp_min(1e-8)), step)
                rate = (step - step_start) / (time.perf_counter() - t_start)
                mem = (torch.cuda.max_memory_allocated() / 2**30
                       if torch.cuda.is_available() else 0.0)
                print(f"step {step:7d} | " +
                      " ".join(f"{k} {v / LOG_EVERY:.4f}" for k, v in running.items()) +
                      f" | {rate:.2f} it/s | peak {mem:.1f} GiB")
                running = {}

            if step % VAL_EVERY == 0:
                # three axes, three prefixes, never averaged - their reference sets
                # differ in size and kind, so a merged number is a weighted average
                # of two different questions
                vm = run_validation(model, val_loader)
                gm = run_validation(model, valgt_loader)
                tm = run_validation(model, train_probe_loader)
                for prefix, m in (("val", vm), ("valgt", gm), ("trainset", tm)):
                    for k, v in m.items():
                        writer.add_scalar(f"{prefix}/{k}", v, step)
                writer.add_scalar("gap/onset_f1", tm["onset_f1"] - vm["onset_f1"], step)
                writer.add_scalar("gap/onset_delay_ms",
                                  vm["onset_delay_ms"] - tm["onset_delay_ms"], step)
                # GOAT-holdout vs unseen-chain electric DI: the gap the new corpora
                # were ingested to close
                writer.add_scalar("gap/domain_onset_f1",
                                  vm["onset_f1"] - gm["onset_f1"], step)
                print(f"  VAL   step {step}: loss {vm['val_loss']:.4f} | "
                      f"onset F1 {vm['onset_f1']:.3f} "
                      f"(P {vm['onset_p']:.3f} R {vm['onset_r']:.3f}) | "
                      f"delay {vm['onset_delay_ms']:+.1f} ms "
                      f"(p90 {vm['onset_delay_ms_p90']:.1f}) | "
                      f"note+offset F1 {vm['note_f1_offset']:.3f} "
                      f"(offset cost {vm['offset_cost']:.3f}, "
                      f"dur x{vm['duration_ratio']:.2f}) | "
                      f"roll F1 {vm['frame_activity_f1']:.3f}")
                print(f"  VALGT step {step}: onset F1 {gm['onset_f1']:.3f} "
                      f"(P {gm['onset_p']:.3f} R {gm['onset_r']:.3f}) | "
                      f"delay {gm['onset_delay_ms']:+.1f} ms "
                      f"(p90 {gm['onset_delay_ms_p90']:.1f}) | "
                      f"roll F1 {gm['frame_activity_f1']:.3f}   "
                      f"<- unseen player+guitar+chain, electric DI; compare "
                      f"against ITS OWN history, not against VAL")
                print(f"  SEEN  step {step}: onset F1 {tm['onset_f1']:.3f} "
                      f"(gap {tm['onset_f1'] - vm['onset_f1']:+.3f}) | "
                      f"delay {tm['onset_delay_ms']:+.1f} ms "
                      f"(p90 {tm['onset_delay_ms_p90']:+.1f}) | "
                      f"roll F1 {tm['frame_activity_f1']:.3f}"
                      + ("   <- ANTICIPATION: firing before the attack on "
                         "seen audio is memorized sequence"
                         if tm["onset_delay_ms"] < 0 else ""))
                if vm["onset_f1"] > best_f1:
                    best_f1 = vm["onset_f1"]
                    torch.save({"model": model.state_dict(), "cfg": cfg.to_dict(),
                                "norm": norm, "step": step, "val": vm,
                                "valgt": gm, "trainset": tm},
                               os.path.join(CKPT_DIR, f"{RUN_NAME}_best.pt"))
                    print(f"  saved best (onset F1 {best_f1:.3f})")

            if step % CKPT_EVERY == 0:
                torch.save({"model": model.state_dict(), "opt": opt.state_dict(),
                            "sched": sched.state_dict(), "cfg": cfg.to_dict(),
                            "norm": norm, "step": step},
                           os.path.join(CKPT_DIR, f"{RUN_NAME}_last.pt"))

            if step >= TOTAL_STEPS:
                break
except KeyboardInterrupt:
    # catching it here is half the fix: IPython never gets to store a traceback,
    # so no frame keeps the activation graph alive
    print(f"\ninterrupted at step {step}")
finally:
    free_interrupted_graph()

## 7. Streaming parity and long-session drift

Two checks no offline metric covers, and both must pass before any C++ work starts.

**Parity:** batched and streamed inference must produce the same numbers. If they diverge,
every metric above is describing a model that is not the one which will run live.

**Drift:** the GRU state is the only unbounded thing in the model. The conv trunk forgets
everything past its receptive field by construction; a recurrent state does not, and can
wander over a set that runs an hour. Windowed validation cannot see this, because it only
ever runs 4 s from a cold start.

In [ ]:
# the twin has no streaming path
if not cfg.causal:
    print("non-causal twin: no streaming path, streaming parity check skipped")
else:
    model.eval()
    probe = torch.randn(1, cfg.hop_length * 200, device=DEVICE) * 0.05

    with torch.no_grad():
        batched = model(probe)
        model.start_session(DEVICE)
        chunks = [model(probe[:, i * cfg.hop_length:(i + 1) * cfg.hop_length])
                  for i in range(200)]
        model.end_session()

    for k in batched:
        streamed = torch.cat([c[k] for c in chunks], dim=-1)
        print(f"stream vs batch {k:9s} max abs err "
              f"{(streamed - batched[k]).abs().max().item():.3e}")

In [ ]:
# the twin has no streaming path
if not cfg.causal:
    print("non-causal twin: no streaming path, long-session drift check skipped")
else:
    entry = max(val_entries, key=lambda e: e["n_samples"])
    audio = np.load(entry["audio"], mmap_mode="r")
    minutes = min(20, int(len(audio) / cfg.sample_rate / 60))
    print(f"streaming {minutes} min of {entry['item']}...")

    CH = cfg.hop_length * 128
    model.eval()
    model.start_session(DEVICE)
    per_min, acc, cur = [], [], 0
    with torch.no_grad():
        n = minutes * 60 * cfg.sample_rate
        for i in range(0, n - CH, CH):
            x = torch.from_numpy(np.asarray(audio[i:i + CH], dtype=np.float32))
            o = model(x.unsqueeze(0).to(DEVICE))
            acc.append(float(torch.sigmoid(o["onset"]).mean()))
            cur += CH
            if cur >= 60 * cfg.sample_rate:
                per_min.append(float(np.mean(acc)))
                acc, cur = [], 0
    model.end_session()

    print("mean onset activation by minute:")
    print("  " + "  ".join(f"{v:.4f}" for v in per_min))
    if len(per_min) >= 4:
        a, b = np.mean(per_min[:2]), np.mean(per_min[-2:])
        verdict = "STABLE" if abs(b - a) < 0.25 * max(a, 1e-6) else "DRIFTING - investigate"
        print(f"first 2 min {a:.4f} vs last 2 min {b:.4f} -> {verdict}")

## 8. Listen to it

Writes predictions on a validation recording to a MIDI file. Load it next to the reference
and listen: an aggregate F1 will never tell you that the model drops the low E under a
chord, or that it double-triggers on palm mutes.

In [ ]:
import pretty_midi

SECONDS = 30
entry = val_entries[1]
audio = np.load(entry["audio"], mmap_mode="r")
x = torch.from_numpy(np.asarray(audio[:SECONDS * cfg.sample_rate], dtype=np.float32))
x = x[: (len(x) // cfg.hop_length) * cfg.hop_length].unsqueeze(0).to(DEVICE)

model.eval()
with torch.no_grad():
    out = {k: v.float() for k, v in model(x).items()}
events = metrics.decode_events(out, cfg)[0]
# host-side loudness: the same heuristic the C++ tool ships (no velocity head)
metrics.annotate_velocity(events, x[0].cpu().numpy(), cfg)
print(f"{len(events)} notes decoded from {SECONDS}s of {entry['item']}")

# one instrument: pitch-major events carry no string
pm = pretty_midi.PrettyMIDI()
inst = pretty_midi.Instrument(program=27, name="gdi2m")
for e in events:
    t0_ = e["frame"] / cfg.frame_rate
    t1_ = max(e["offset"], e["frame"] + 1) / cfg.frame_rate   # decoded release
    inst.notes.append(pretty_midi.Note(
        velocity=e["velocity"],
        pitch=e["pitch"], start=t0_, end=t1_))
pm.instruments.append(inst)
durs = [(e["offset"] - e["frame"]) / cfg.frame_rate for e in events]
print(f"decoded note length: median {1000 * np.median(durs):.0f} ms" if durs else "")
out_mid = os.path.join(CKPT_DIR, f"preview_{entry['item']}.mid")
pm.write(out_mid)
print("wrote", out_mid)

ref = np.load(entry["events"])
print(f"reference has {len(ref[ref['onset'] < SECONDS])} notes in the same span")

In [ ]:
from IPython.display import Audio, display

# same truncation the preview used, so this is exactly what the model was fed
n = (SECONDS * cfg.sample_rate // cfg.hop_length) * cfg.hop_length
clip = np.asarray(audio[:n], dtype=np.float32)
peak = float(np.abs(clip).max())
rms = float(np.sqrt((clip ** 2).mean()))

print(f"{entry['item']}: {n / cfg.sample_rate:.1f}s DI at {cfg.sample_rate} Hz | "
      f"player {entry['player']}, {entry['guitar']}, {entry['tuning_class']}")
print(f"peak {peak:.3f}  rms {rms:.4f}"
      + ("   <- quiet; pass normalize=True if you cannot hear it" if peak < 0.15 else ""))

display(Audio(np.clip(clip, -1.0, 1.0), rate=cfg.sample_rate, normalize=False))

In [ ]:
import soundfile as sf
import pretty_midi
from math import gcd
from scipy.signal import resample_poly
from IPython.display import Audio, display

WAV = os.path.join(PROJECT, "testsong.wav")
ONSET_THRESHOLD = cfg.onset_threshold

raw, sr_in = sf.read(WAV, dtype="float32", always_2d=True)
ch_rms = np.sqrt((raw ** 2).mean(0))
if len(ch_rms) > 1 and ch_rms.min() < 0.1 * ch_rms.max():
    ch = int(np.argmax(ch_rms))
    x, how = raw[:, ch], f"channel {ch} only (other is near-silent)"
else:
    x, how = raw.mean(1), "channels averaged"

g = gcd(sr_in, cfg.sample_rate)
x = resample_poly(x, cfg.sample_rate // g, sr_in // g).astype(np.float32)
n = (len(x) // cfg.hop_length) * cfg.hop_length
x = np.ascontiguousarray(x[:n])
print(f"{sr_in} Hz -> {cfg.sample_rate} Hz ({cfg.sample_rate // g}/{sr_in // g}), "
      f"{how}, {n / cfg.sample_rate:.1f}s, peak {np.abs(x).max():.3f}")

# streamed, not batched: bounded memory regardless of length, and it exercises
# the same path the plugin will take; the twin has no streaming path
was_training = model.training
model.eval()
with torch.no_grad():
    if cfg.causal:
        model.start_session(DEVICE)
        CHUNK, outs = cfg.hop_length * 128, []
        for i in range(0, n, CHUNK):
            c = torch.from_numpy(x[i:i + CHUNK])
            c = c[: (len(c) // cfg.hop_length) * cfg.hop_length]
            if not len(c):
                break
            o = model(c.unsqueeze(0).to(DEVICE))
            outs.append({k: v.float().cpu() for k, v in o.items()})
        model.end_session()
        out = {k: torch.cat([o[k] for o in outs], -1) for k in outs[0]}
    else:
        out = {k: v.float().cpu()
               for k, v in model(torch.from_numpy(x).unsqueeze(0).to(DEVICE)).items()}
if was_training:
    model.train()                      # so training resumes cleanly

# how sensitive is the threshold on YOUR audio? pick from this, do not guess
probs = torch.sigmoid(out["onset"])
print("onsets vs threshold:", {round(t, 2): int((probs > t).sum())
                               for t in (0.2, 0.3, 0.5, 0.7, 0.9)})

events = metrics.decode_events(out, cfg, onset_threshold=ONSET_THRESHOLD)[0]
metrics.annotate_velocity(events, x, cfg)
events = [e for e in events if e["frame"] > cfg.loss_cutoff]   # drop cold-start frames
print(f"\n{len(events)} notes "
      f"({len(events) / (n / cfg.sample_rate) * 60:.0f} per minute)")
if events:
    ps = [e["pitch"] for e in events]
    vs = [e["velocity"] for e in events]
    print(f"pitch range MIDI {min(ps)}..{max(ps)}  "
          f"velocity {min(vs)}..{max(vs)} (host-side loudness meter)")

pm = pretty_midi.PrettyMIDI()
inst = pretty_midi.Instrument(program=27, name="gdi2m")
for e in events:
    inst.notes.append(pretty_midi.Note(
        velocity=e["velocity"], pitch=e["pitch"],
        start=e["frame"] / cfg.frame_rate,
        end=max(e["offset"], e["frame"] + 1) / cfg.frame_rate))
pm.instruments.append(inst)
dest = os.path.join(PROJECT, "testsong_transcribed.mid")
pm.write(dest)
print("wrote", dest)

display(Audio(np.clip(x, -1.0, 1.0), rate=cfg.sample_rate, normalize=False))

In [ ]:
torch.cuda.empty_cache()